In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

# ============================
# 1. CARICAMENTO FILE
# ============================
ambientale = pd.read_excel(r"../INDICI 2021/dataset_macroarea_ambientale_2021.xlsx")
socio      = pd.read_excel(r"../INDICI 2021/dataset_macroarea_socio_2021.xlsx")
economico  = pd.read_excel(r"../INDICI 2021/dataset_macroarea_economica_2021.xlsx")
servizi    = pd.read_excel(r"../INDICI 2021/dataset_macroarea_servizi_2021.xlsx")

# ============================
# 2. POLARITÀ
# ============================
direction = {
    # AMBIENTE
    "YACAQI": 1,
    "consumo_del_suolo": 1,
    "raccolta_indifferenziata_dei_rifiuti_urbani_per_abitante": 1,
    "tasso_di_motorizzazione_ad_alta_emissione": 1,
    "rischio_num": 1,
    "scarto_aree_protette": -1,

    # SOCIO
    "ind_vecchiaia_21": 1,
    "quota_pop_attiva_21": -1,
    "tasso_crescita_naturale_2021": -1,
    "Tasso migratorio totale 2021": 1,
    "Stranieri/pop 2021": -1,
    "Densità abitativa 2021": -1,

    # ECONOMICO
    "reddito_medio_per_contribuente_2021": -1,
    "% Addetti alle imprese su popolazione 2021": -1,
    "imprese_per_1000_ab": -1,
    "laureati_per_1000_ab": -1,
    "Contribuenti IRPEF con reddito complessivo inferiore a 10.000 euro nel 2021 (%)": 1,

    # SERVIZI
    "indice_di_accessibilita_ai_servizi_essenziali": 1,
    "sportelli_dev": -1,
    "musei_dev": -1,
    "sanita_dev": -1,
    "Superficie esercizi commerciali per kmq 2021": -1,
    "Addetti dei servizi alla persona per 1.000 abitanti nel 2021": -1
}

# ============================
# 3. PREPROCESSING
# ============================
def preprocess(df, direction):
    
    df = df.copy()
    df["pro_com"] = df["pro_com"].astype(str)
    
    X = df.select_dtypes(include=[np.number]).copy()
    
    # polarità
    for col in X.columns:
        if col in direction:
            X[col] = X[col] * direction[col]
    
    # rimozione NA
    mask = X.notna().all(axis=1)
    X = X[mask]
    ids = df.loc[mask, "pro_com"]
    
    # standardizzazione
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    return X_scaled, ids

X_amb, id_amb = preprocess(ambientale, direction)
X_soc, id_soc = preprocess(socio, direction)
X_eco, id_eco = preprocess(economico, direction)
X_ser, id_ser = preprocess(servizi, direction)

# ============================
# 4. AUTOENCODER
# ============================
def autoencoder_index(X):
    
    input_dim = X.shape[1]
    
    input_layer = Input(shape=(input_dim,))
    
    encoded = Dense(input_dim // 2, activation="relu")(input_layer)
    bottleneck = Dense(1, activation="linear")(encoded)
    
    decoded = Dense(input_dim // 2, activation="relu")(bottleneck)
    output_layer = Dense(input_dim, activation="linear")(decoded)
    
    autoencoder = Model(inputs=input_layer, outputs=output_layer)
    autoencoder.compile(optimizer="adam", loss="mse")
    
    autoencoder.fit(X, X, epochs=100, batch_size=32, verbose=0)
    
    encoder = Model(inputs=input_layer, outputs=bottleneck)
    
    index = encoder.predict(X).flatten()
    
    return index

# ============================
# 5. COSTRUZIONE INDICI
# ============================
idx_amb = autoencoder_index(X_amb)
idx_soc = autoencoder_index(X_soc)
idx_eco = autoencoder_index(X_eco)
idx_ser = autoencoder_index(X_ser)

# ============================
# 6. ALLINEAMENTO SEGNO
# ============================
def align_sign(idx, X):
    if np.corrcoef(idx, X.mean(axis=1))[0,1] < 0:
        return -idx
    return idx

idx_amb = align_sign(idx_amb, X_amb)
idx_soc = align_sign(idx_soc, X_soc)
idx_eco = align_sign(idx_eco, X_eco)
idx_ser = align_sign(idx_ser, X_ser)

# ============================
# 7. NORMALIZZAZIONE
# ============================
def norm01(x):
    return (x - np.min(x)) / (np.max(x) - np.min(x))

idx_amb = norm01(idx_amb)
idx_soc = norm01(idx_soc)
idx_eco = norm01(idx_eco)
idx_ser = norm01(idx_ser)

# ============================
# 8. MERGE
# ============================
df_indices = (
    pd.DataFrame({"pro_com": id_amb, "indice_ambientale": idx_amb})
    .merge(pd.DataFrame({"pro_com": id_soc, "indice_socio": idx_soc}, ), on="pro_com")
    .merge(pd.DataFrame({"pro_com": id_eco, "indice_economico": idx_eco}), on="pro_com")
    .merge(pd.DataFrame({"pro_com": id_ser, "indice_servizi": idx_ser}), on="pro_com")
)

# ============================
# 9. EXPORT
# ============================
df_indices.to_excel(
    r"../INDICI 2021/dataset_indici_autoencoder_2021.xlsx",
    index=False
)

print("✅ FATTO: indici autoencoder 2021 salvati")